<a href="https://colab.research.google.com/github/Takayoshi-code/nanoGPT-class/blob/master/nanoGPT_Aozora_tiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP1 nanoGPT環境をGitHubからダウンロードして自分の環境に展開します。

In [ ]:
# ===== 安全版 =====

import os

print("===== Step 0: /content に移動 =====")
%cd /content

print("===== Step 1: クリーン =====")
!rm -rf nanoGPT
!rm -f master.zip

print("===== Step 2: ライブラリ =====")
!pip install -q sentencepiece datasets tqdm

print("===== Step 3: nanoGPT取得 =====")

!wget -q https://github.com/Takayoshi-code/nanoGPT-class/archive/refs/heads/master.zip
!unzip -q master.zip
!mv nanoGPT-class-master nanoGPT

print("===== Step 4: 移動 =====")
%cd /content/nanoGPT

print("===== Step 5: 確認 =====")
!ls

===== Step 0: /content に移動 =====
/content
===== Step 1: クリーン =====
===== Step 2: ライブラリ =====
===== Step 3: nanoGPT取得 =====
===== Step 4: 移動 =====
/content/nanoGPT
===== Step 5: 確認 =====
assets	  configurator.py  model.py   scaling_laws.ipynb
bench.py  data		   README.md  train.py
config	  LICENSE	   sample.py  transformer_sizing.ipynb


## STEP2 青空文庫から好きな小説をダウンロードします。

In [ ]:
import os
import re
import csv
import zipfile
import urllib.request
from wcwidth import wcswidth

# =====================================================
# 設定
# =====================================================

TARGET_AUTHOR = "太宰治"

TARGET_TITLES = {
   "人間失格",
   #"グッド・バイ",
   # "ヴィヨンの妻",
   # "津軽",
   # "斜陽",
   #"走れメロス"
   # "きりぎりす",
   # "女生徒",
   # "お伽草紙",
   # "東京八景",
   # "天狗",
   # "葉",
   # "パウロの混乱",
   # "薄明",
   # "葉桜と魔笛",
   # "恥",
   # "走ラヌ名馬",
   # "八十八夜",
   # "花火",
   # "花吹雪",
   # "母",
   # "春",
   # "春の枯葉",
   # "春の盗賊",
   # "パンドラの匣",
   # "犯人",
   # "晩年",
   # "眉山",
   # "美少女",
   # "美男子と煙草",
   # "一つの約束",
   # "火の鳥",
   # "皮膚と心"
}

MIN_CHARS = 3000

OUT_DIR = "aozora_selected"
os.makedirs(OUT_DIR, exist_ok=True)

LIST_URL = (
    "https://www.aozora.gr.jp/index_pages/"
    "list_person_all_extended_utf8.zip"
)

LIST_ZIP = "list.zip"
CSV_FILE = "list_person_all_extended_utf8.csv"

# =====================================================
# 日本語表示
# =====================================================

def jp_pad(text, width):
    pad = width - wcswidth(text)
    return text + " " * max(0, pad)

# =====================================================
# 青空文庫クリーニング
# =====================================================

def clean_text(text):

    text = text.replace("\r", "")

    # -----------------------------
    # ヘッダ除去
    # -----------------------------
    if "-------------------------------------------------------" in text:
        parts = text.split(
            "-------------------------------------------------------"
        )

        if len(parts) >= 3:
            text = (
                "-------------------------------------------------------"
                .join(parts[2:])
            )

    # -----------------------------
    # フッタ除去
    # -----------------------------
    end_markers = [
        "底本：",
        "入力：",
        "校正：",
        "このファイルは、",
        "青空文庫作成ファイル："
    ]

    cut_pos = len(text)

    for marker in end_markers:

        pos = text.find(marker)

        if pos != -1:
            cut_pos = min(cut_pos, pos)

    text = text[:cut_pos]

    # -----------------------------
    # ルビ除去
    # -----------------------------
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"｜", "", text)

    # -----------------------------
    # 注記除去
    # -----------------------------
    text = re.sub(r"［＃.*?］", "", text)

    # -----------------------------
    # 区切り線除去
    # -----------------------------
    text = re.sub(r"-{5,}", "", text)

    # -----------------------------
    # 空行整理
    # -----------------------------
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

# =====================================================
# 青空文庫リスト取得
# =====================================================

if not os.path.exists(LIST_ZIP):

    print("Downloading Aozora list...")

    urllib.request.urlretrieve(
        LIST_URL,
        LIST_ZIP
    )

if not os.path.exists(CSV_FILE):

    with zipfile.ZipFile(LIST_ZIP) as z:
        z.extractall()

# =====================================================
# CSV読み込み
# =====================================================

with open(
    CSV_FILE,
    encoding="utf-8-sig"
) as f:

    reader = csv.reader(f)

    header = next(reader)

    last_name_idx = header.index("姓")
    first_name_idx = header.index("名")
    title_idx = header.index("作品名")
    url_idx = header.index("テキストファイルURL")

    rows = list(reader)

print(f"CSV loaded : {len(rows):,}")

# =====================================================
# ダウンロード
# =====================================================

corpus = []

stats = []

downloaded_titles = set()

for row in rows:

    try:

        title = row[title_idx]

        if title not in TARGET_TITLES:
            continue

        author = (
            row[last_name_idx]
            + row[first_name_idx]
        )

        if author != TARGET_AUTHOR:
            continue

        if title in downloaded_titles:
            continue

        url = row[url_idx]

        if not url.endswith(".zip"):
            continue

        print(
            f"Downloading : "
            f"{title} ({author})"
        )

        urllib.request.urlretrieve(
            url,
            "tmp.zip"
        )

        with zipfile.ZipFile("tmp.zip") as z:

            txt_files = [
                x for x in z.namelist()
                if x.lower().endswith(".txt")
            ]

            if len(txt_files) == 0:
                continue

            raw = z.read(txt_files[0])

            try:
                text = raw.decode("shift_jis")
            except:
                text = raw.decode(
                    "utf-8",
                    errors="ignore"
                )

            text = clean_text(text)

            if len(text) < MIN_CHARS:
                continue

            downloaded_titles.add(title)

            corpus.append(text)

            chars = len(text)
            size_bytes = len(
                text.encode("utf-8")
            )

            stats.append({
                "title": title,
                "author": author,
                "chars": chars,
                "kb": size_bytes / 1024,
                "mb": size_bytes / 1024 / 1024
            })

        os.remove("tmp.zip")

    except Exception as e:

        print(
            f"ERROR : {title}"
        )

        print(e)

        if os.path.exists("tmp.zip"):
            os.remove("tmp.zip")

# =====================================================
# 保存
# =====================================================

outfile = os.path.join(
    OUT_DIR,
    "selected_works.txt"
)

with open(
    outfile,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n<|endoftext|>\n".join(corpus)
    )

# =====================================================
# 統計
# =====================================================

print()
print("=" * 90)
print("DOWNLOADED WORKS")
print("=" * 90)

print(
    jp_pad("作品名", 30),
    jp_pad("作者", 15),
    "文字数".rjust(12),
    "KB".rjust(12),
    "MB".rjust(12)
)

print("-" * 90)

total_chars = 0
total_bytes = 0

for s in sorted(
    stats,
    key=lambda x: x["title"]
):

    total_chars += s["chars"]
    total_bytes += int(
        s["kb"] * 1024
    )

    print(
        jp_pad(s["title"], 30),
        jp_pad(s["author"], 15),
        f"{s['chars']:,}".rjust(12),
        f"{s['kb']:.1f}".rjust(12),
        f"{s['mb']:.3f}".rjust(12)
    )

print("-" * 90)

print(
    f"作品数       : {len(stats)}"
)

print(
    f"総文字数     : {total_chars:,}"
)

print(
    f"総サイズ     : "
    f"{total_bytes/1024:.1f} KB "
    f"({total_bytes/1024/1024:.3f} MB)"
)

outfile_size = os.path.getsize(outfile)

print()
print(f"保存先       : {outfile}")

print(
    f"コーパス容量 : "
    f"{outfile_size/1024:.1f} KB "
    f"({outfile_size/1024/1024:.3f} MB)"
)

print("=" * 90)
print("DONE")
print("=" * 90)

CSV loaded : 19,502

DOWNLOADED WORKS
作品名                         作者                     文字数           KB           MB
------------------------------------------------------------------------------------------
人間失格                       太宰治                73,849        214.7        0.210
------------------------------------------------------------------------------------------
作品数       : 1
総文字数     : 73,849
総サイズ     : 214.7 KB (0.210 MB)

保存先       : aozora_selected/selected_works.txt
コーパス容量 : 214.7 KB (0.210 MB)
DONE


## STEP3 入力コーパスのクリーニングをします。

In [ ]:
# ============================================================
# 太宰治_5works.txt 強力クリーニング版
# ============================================================

import os
import re
from tqdm import tqdm

# ============================================================
# 入出力
# ============================================================

#INPUT = "aozora_selected/selected_works.txt"
#OUTPUT = "aozora_selected/selected_works_clean.txt"
INPUT = "/content/nanoGPT/aozora_selected/selected_works.txt"
OUTPUT = "/content/nanoGPT/aozora_selected/selected_works_clean.txt"

# ============================================================
# クリーニング
# ============================================================

def clean_text(text):

    lines = text.split("\n")

    result = []
    started = False

    for line in lines:

        raw = line
        line = line.strip()

        # --------------------------------
        # title / author 削除
        # --------------------------------
        if line.startswith("<|title|>"):
            continue

        if line.startswith("<|author|>"):
            continue

        # --------------------------------
        # 本文開始判定
        # --------------------------------
        if not started:

            if (
                raw.startswith("　")
                and len(line) > 10
                and "。" in line
            ):
                started = True
            else:
                continue

        # --------------------------------
        # フッタ検出
        # --------------------------------
        if (
            line.startswith("底本")
            or line.startswith("初出")
            or line.startswith("入力")
            or line.startswith("校正")
            or line.startswith("青空文庫")
            or line.startswith("作成ファイル")
            or "青空文庫作成ファイル" in line
        ):
            break

        # --------------------------------
        # 注記/JIS
        # --------------------------------
        if (
            "記号について" in line
            or "JIS" in line
            or "入力に使用" in line
            or "校正に使用" in line
        ):
            continue

        # --------------------------------
        # 見出し
        # --------------------------------
        if re.fullmatch(
            r"[一二三四五六七八九十百千万]+",
            line
        ):
            continue

        if re.fullmatch(
            r"[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+",
            line
        ):
            continue

        if line in {
            "上",
            "中",
            "下",
            "前編",
            "後編",
            "序",
            "跋",
            "附録",
            "解説"
        }:
            continue

        # --------------------------------
        # ルビ
        # --------------------------------
        line = re.sub(r"｜", "", line)
        line = re.sub(r"《.*?》", "", line)
        line = re.sub(r"［＃.*?］", "", line)

        # --------------------------------
        # 注釈
        # --------------------------------
        line = re.sub(r"〔.*?〕", "", line)
        line = re.sub(r"（注.*?）", "", line)

        # --------------------------------
        # ページ番号
        # --------------------------------
        line = re.sub(
            r"（[\d０-９]+）",
            "",
            line
        )

        # --------------------------------
        # 不要記号
        # --------------------------------
        line = line.replace("／＼", "")
        line = line.replace("※", "")
        line = line.replace("＊", "")

        # --------------------------------
        # 英数字のみ
        # --------------------------------
        if re.fullmatch(
            r"[0-9０-９A-Za-z]+",
            line
        ):
            continue

        # --------------------------------
        # 短すぎる行
        # --------------------------------
        if len(line) < 2:
            continue

        result.append(line)

    text = "\n".join(result)

    # --------------------------------
    # 空行整理
    # --------------------------------
    text = re.sub(r"\n{3,}", "\n\n", text)

    # --------------------------------
    # 品質判定
    # --------------------------------
    if text.count("。") < 10:
        return ""

    if len(text) < 1000:
        return ""

    return text.strip()

# ============================================================
# メイン
# ============================================================

def main():

    with open(INPUT, "r", encoding="utf-8", errors="ignore") as f:
        data = f.read()

    original_size = len(data.encode("utf-8"))

    chunks = re.split(r"<\|endoftext\|>", data)

    total = 0
    kept = 0

    total_chars = 0

    with open(OUTPUT, "w", encoding="utf-8") as out:

        for chunk in tqdm(chunks):

            chunk = chunk.strip()

            if not chunk:
                continue

            total += 1

            cleaned = clean_text(chunk)

            if cleaned:

                chars = len(cleaned)

                print(
                    f"Work {kept+1:2d}: "
                    f"{chars:,} chars"
                )

                total_chars += chars

                out.write(cleaned)
                out.write("\n\n<|endoftext|>\n\n")

                kept += 1

    cleaned_size = os.path.getsize(OUTPUT)

    print()
    print("====================================")
    print(f"Total works      : {total}")
    print(f"Kept works       : {kept}")
    print(f"Removed works    : {total-kept}")
    print("------------------------------------")
    print(f"Total chars      : {total_chars:,}")
    print("------------------------------------")
    print(
        f"Original size    : "
        f"{original_size/1024:.1f} KB"
    )
    print(
        f"Cleaned size     : "
        f"{cleaned_size/1024:.1f} KB"
    )
    print(
        f"Reduction        : "
        f"{100*(1-cleaned_size/original_size):.1f}%"
    )
    print("------------------------------------")
    print(f"Output           : {OUTPUT}")
    print("====================================")

# ============================================================

if __name__ == "__main__":
    main()

100%|██████████| 1/1 [00:00<00:00, 119.48it/s]

Work  1: 73,365 chars

Total works      : 1
Kept works       : 1
Removed works    : 0
------------------------------------
Total chars      : 73,365
------------------------------------
Original size    : 214.7 KB
Cleaned size     : 213.4 KB
Reduction        : 0.6%
------------------------------------
Output           : /content/nanoGPT/aozora_selected/selected_works_clean.txt


## STEP4 SentencePiece（5000語）規模のトークナイザを動かしinput.textに登場するトークンを学習する。

In [ ]:
# ============================================================
# 日本語コーパス → SentencePiece → train.bin（最終完成版）
# ============================================================

import os
import re
import sentencepiece as spm
import numpy as np
import pickle
import threading
import time

BASE = "/content/nanoGPT"
INPUT = "/content/nanoGPT/aozora_selected/selected_works_clean.txt"
MODEL_PREFIX = "/content/nanoGPT/jp"
OUT_DIR = os.path.join(BASE, "jp")

# =========================
# 全体タイマー
# =========================
total_start = time.time()

# =========================
# 進捗表示
# =========================
running = True
def heartbeat():
    flag = True
    while running:
        print("ただいま計算中..." if flag else "　　　　　　　　", flush=True)
        flag = not flag
        time.sleep(5)

t = threading.Thread(target=heartbeat)
t.start()

try:
    # =========================
    # ① モデル削除
    # =========================
    t0 = time.time()
    for ext in [".model", ".vocab"]:
        path = MODEL_PREFIX + ext
        if os.path.exists(path):
            os.remove(path)
    print(f"[STEP1] モデル削除時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ② tokenizer学習
    # =========================
    print("=== SentencePiece Training ===")
    t0 = time.time()

    spm.SentencePieceTrainer.train(
        input=INPUT,
        model_prefix=MODEL_PREFIX,
        vocab_size=5000,
        character_coverage=0.9995,
        model_type='unigram',
        num_threads=8,
        input_sentence_size=2000000,   # ←増やした
        shuffle_input_sentence=True,
        max_sentence_length=4096,
        hard_vocab_limit=False,
        user_defined_symbols=["<|endoftext|>"]
    )

    print(f"[STEP2] Tokenizer学習時間: {time.time() - t0:.2f} 秒")
    print("Tokenizer DONE")

    # =========================
    # ③ トークナイズ（最重要部分）
    # =========================
    print("=== Encoding ===")
    t0 = time.time()

    sp = spm.SentencePieceProcessor()
    sp.load(MODEL_PREFIX + ".model")

    ids = []
    eot_id = sp.piece_to_id("<|endoftext|>")

    # 全読み込み
    with open(INPUT, encoding="utf-8") as f:
        data = f.read()

    # 元chunk分割
    chunks = data.split("<|endoftext|>")

    # -------------------------
    # 文単位に細分化（超重要）
    # -------------------------
    new_chunks = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        # 文分割（精度UP）
        sentences = re.split(r"[。！？]", chunk)

        for s in sentences:
            s = s.strip()
            if len(s) > 50:
                new_chunks.append(s + "。")

    # -------------------------
    # encode
    # -------------------------
    for chunk in new_chunks:
        ids.extend(sp.encode(chunk))
        ids.append(eot_id)

    print("Total tokens:", len(ids))
    print(f"[STEP3] Encoding時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ④ train / val 分割
    # =========================
    t0 = time.time()

    n = int(len(ids) * 0.9)
    train_ids = np.array(ids[:n], dtype=np.uint16)
    val_ids   = np.array(ids[n:], dtype=np.uint16)

    print(f"[STEP4] 分割時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ⑤ 保存
    # =========================
    t0 = time.time()

    os.makedirs(OUT_DIR, exist_ok=True)

    train_ids.tofile(os.path.join(OUT_DIR, "train.bin"))
    val_ids.tofile(os.path.join(OUT_DIR, "val.bin"))

    with open(os.path.join(OUT_DIR, "meta.pkl"), "wb") as f:
        pickle.dump({"vocab_size": sp.get_piece_size()}, f)

    print(f"[STEP5] 保存時間: {time.time() - t0:.2f} 秒")

    print("=== ALL DONE ===")
    print("vocab_size =", sp.get_piece_size())

finally:
    running = False
    t.join()

# =========================
# 総時間
# =========================
print(f"\n=== 総処理時間: {time.time() - total_start:.2f} 秒 ===")

ただいま計算中...
[STEP1] モデル削除時間: 0.00 秒
=== SentencePiece Training ===
[STEP2] Tokenizer学習時間: 0.29 秒
Tokenizer DONE
=== Encoding ===
Total tokens: 27336
[STEP3] Encoding時間: 0.02 秒
[STEP4] 分割時間: 0.00 秒
[STEP5] 保存時間: 0.00 秒
=== ALL DONE ===
vocab_size = 5000

=== 総処理時間: 5.00 秒 ===


## STEP5 入力コーパスを学習用と検証用の２つに分けます。

In [ ]:
import sentencepiece as spm
import numpy as np
import os
import pickle

print("=== SentencePiece Training ===")


BASE = "/content/nanoGPT"
sp = spm.SentencePieceProcessor()
sp.load(os.path.join(BASE, "jp.model"))


input_file = "/content/nanoGPT/aozora_selected/selected_works.txt"
#out_dir = os.path.join(BASE, "jp")
out_dir = os.path.join(BASE, "data", "jp")

os.makedirs(out_dir, exist_ok=True)

print("Reading...")
ids = []
with open(input_file, encoding="utf-8") as f:
    for line in f:
        ids.extend(sp.encode(line))   # ← メモリ安全

print("Encoding done")
print("Total tokens:", len(ids))

n = int(0.9 * len(ids))
train_ids = np.array(ids[:n], dtype=np.uint16)
val_ids = np.array(ids[n:], dtype=np.uint16)

train_ids.tofile(os.path.join(out_dir, "train.bin"))
val_ids.tofile(os.path.join(out_dir, "val.bin"))

with open(os.path.join(out_dir, "meta.pkl"), "wb") as f:
    pickle.dump({"vocab_size": sp.get_piece_size()}, f)
print(f"train tokens = {len(train_ids):,}")
print(f"val tokens   = {len(val_ids):,}")

print("DONE")

=== SentencePiece Training ===
Reading...
Encoding done
Total tokens: 36520
train tokens = 32,868
val tokens   = 3,652
DONE


## STEP6 GPUが使えるかをチェックします。

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## STEP 7 Transformerで学習させます。　CPU環境では --device=cpu とし、 GPUが使えるなら　--device=cuda にする。

In [ ]:
## 人間失格規模用

!python train.py \
  --dataset=jp \
  --device=cuda \
  --compile=False \
  --init_from=scratch \
  --n_layer=22 \
  --n_head=16 \
  --n_embd=128 \
  --batch_size=8 \
  --gradient_accumulation_steps=1 \
  --block_size=1024 \
  --max_iters=8000 \
  --lr_decay_iters=8000 \
  --warmup_iters=20 \
  --learning_rate=4e-4 \
  --min_lr=3e-5 \
  --eval_interval=20 \
  --eval_iters=20 \
  --dropout=0.1 \
  --log_interval=10 \
  --dtype=float16


Overriding: dataset = jp
Overriding: device = cuda
Overriding: compile = False
Overriding: init_from = scratch
Overriding: n_layer = 22
Overriding: n_head = 16
Overriding: n_embd = 128
Overriding: batch_size = 8
Overriding: gradient_accumulation_steps = 1
Overriding: block_size = 1024
Overriding: max_iters = 20000
Overriding: lr_decay_iters = 20000
Overriding: warmup_iters = 20
Overriding: learning_rate = 0.0004
Overriding: min_lr = 3e-05
Overriding: eval_interval = 20
Overriding: eval_iters = 20
Overriding: dropout = 0.1
Overriding: log_interval = 10
Overriding: dtype = float16
tokens per iteration will be: 8,192
found vocab_size = 5000 (inside data/jp/meta.pkl)
Initializing a new model from scratch
number of parameters: 4.97M
/content/nanoGPT/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 90, wi

In [ ]:
## 走れメロス規模用
!python train.py \
  --dataset=jp \
  --device=cuda \
  --compile=False \
  --init_from=scratch \
  --n_layer=2 \
  --n_head=2 \
  --n_embd=128 \
  --batch_size=4 \
  --gradient_accumulation_steps=1 \
  --block_size=128 \
  --max_iters=6000 \
  --lr_decay_iters=6000 \
  --warmup_iters=20 \
  --learning_rate=3e-4 \
  --min_lr=3e-5 \
  --eval_interval=20 \
  --eval_iters=20 \
  --dropout=0.1 \
  --log_interval=10 \
  --dtype=float16


Overriding: dataset = jp
Overriding: device = cuda
Overriding: compile = False
Overriding: init_from = scratch
Overriding: n_layer = 2
Overriding: n_head = 2
Overriding: n_embd = 128
Overriding: batch_size = 4
Overriding: gradient_accumulation_steps = 1
Overriding: block_size = 128
Overriding: max_iters = 6000
Overriding: lr_decay_iters = 6000
Overriding: warmup_iters = 20
Overriding: learning_rate = 0.0003
Overriding: min_lr = 3e-05
Overriding: eval_interval = 20
Overriding: eval_iters = 20
Overriding: dropout = 0.1
Overriding: log_interval = 10
Overriding: dtype = float16
tokens per iteration will be: 512
found vocab_size = 5000 (inside data/jp/meta.pkl)
Initializing a new model from scratch
number of parameters: 1.03M
/content/nanoGPT/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 10, with 1,04

## STEP8 文章生成をします。

In [ ]:
# generate_sp.py
import os
import torch
import sentencepiece as spm
from contextlib import nullcontext
os.chdir("/content/nanoGPT")
from model import GPTConfig, GPT
# =========================
# 設定
# =========================
out_dir = 'out'
#sp_model_path = 'jp16k.model'
#sp_model_path = "/home/yokota/DAZAI_LLM/nanoGPT/jp.model"
sp_model_path = "/content/nanoGPT/jp.model"


prompt = "女は、甲州の"   # ← ここで文書の方向性を決める
num_samples = 1
max_new_tokens = 200   # ← 長文
#temperature = 0.9     # ← 安定寄り
#top_k = 20
device = 'cuda' if torch.cuda.is_available() else 'cpu'

repetition_penalty = 1.1
temperature = 0.2
top_k = 20
top_p = 0.9
# =========================
# SentencePiece
# =========================
sp = spm.SentencePieceProcessor()
sp.load(sp_model_path)

def encode(s): return sp.encode(s, out_type=int)
def decode(l): return sp.decode(l)

# =========================
# モデル
# =========================
ckpt_path = os.path.join(out_dir, 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)

model = GPT(GPTConfig(**checkpoint['model_args']))
state_dict = checkpoint['model']

# compile対策
for k in list(state_dict.keys()):
    if k.startswith('_orig_mod.'):
        state_dict[k[len('_orig_mod.'):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

ctx = nullcontext()

# =========================
# 生成（改良版）
# =========================
# =========================
# 生成（最終版・完成）
# =========================
def generate(model, idx, max_new_tokens):

    block_size = model.config.block_size
    last_tokens = []  # ← 追加（直近トークン履歴）

    for _ in range(max_new_tokens):

        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
        logits, _ = model(idx_cond)

        logits = logits[:, -1, :] / temperature

        # -------------------------
        # repetition penalty（logitsに直接適用）
        # -------------------------
        for token in set(idx[0].tolist()):
            logits[0][token] /= repetition_penalty

        # -------------------------
        # top_k
        # -------------------------
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')

        # -------------------------
        # top_p（追加）
        # -------------------------
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[0][indices_to_remove] = -float('Inf')

        probs = torch.softmax(logits, dim=-1)

        # -------------------------
        # last_tokens repetition防止（最重要）
        # -------------------------
        for _ in range(10):  # 最大10回リトライ
            next_token = torch.multinomial(probs, num_samples=1)

            if next_token.item() not in last_tokens:
                break

        # 履歴更新
        last_tokens.append(next_token.item())
        if len(last_tokens) > 10:
            last_tokens.pop(0)

        idx = torch.cat((idx, next_token), dim=1)

    return idx
# =========================
# 実行
# =========================
start_ids = encode(prompt)
x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

with torch.no_grad():
    with ctx:
        for _ in range(num_samples):
            y = generate(model, x, max_new_tokens)
            text = decode(y[0].tolist())

            print("==========")
            print(text)

number of parameters: 4.97M
女は、甲州の生れで二十八歳でした。五つになる女児と、高円寺のアパートに住んでいました。夫と死別して、三年になると言っていました。 「あなたは、ずいぶん苦労して育って来たみたいなひとね。よく気がきくわ。可哀そうに」 はじめて、男めかけみたいな生活をしました。シヅ子(というのが、その女記者の名前でした)が新宿の雑誌社に勤めに出たあとは、自分とそれからシゲ子という五つの女児と二人、おとなしくお留守番という事になりました。それまでは、母の留守には、シゲ子はアパートの管理人の部屋で遊んでいたようでしたが、「気のきく」おじさんが遊び相手として現われたので、大いに御機嫌がいい様子でした。 一週間ほど、ぼんやり、自分はそこにいました。アパートの窓のすぐ近くの電線に、奴凧が一つひっからまっていて、春のほこり風に吹かれ、破られ、それでもなかなか、しつっこく電線にからみついて離れず、何やら首肯いたりなんかしているので、自分はそれを見る度毎に苦笑し、
